# Aim 3
Tara Cornwell June 2024

Adapted from Darici and Kuo

In [17]:
# Import necessary packages
using DynLoco_TC, Plots, CSV, DataFrames, Statistics, StructArrays

# Constants
g = 9.81
L = 1
norm_t = (L/g)^0.5
norm_v = (g*L)^0.5
nsteps = 7
pert_mag = 1.5
group = 1

# Clear variables
wstar = nothing

# Nominal gait
if group == 1
    # Nonstroke
    wstar = findgait(WalkRW2lvs(safety=true,beta=0.49), target=:speed=>0.36, varying=:P); # varies SL nonlinearly
    nom_u = optwalk_TC(wstar, nsteps, boundaryvels = (wstar.vm, wstar.vm), boundarywork=false, J="u");
    max_P = 2*ones(nsteps)
else
    # Stroke
    P_NP = 2
    
    if group == 2
        P_P = 0.94 # 20%
    elseif group == 3
        P_P = 0.87 # 40%
    elseif group == 4
        P_P = 0.78 # 60%
    elseif group == 5
        P_P = 2 # 0%
    end
    max_P = [P_NP P_P P_NP P_P P_NP P_P P_NP]
    wstar = findgait(WalkRW2lvs(safety=true,beta=0.49), target=:speed=>0.23, varying=:P); # varies SL nonlinearly
    nom_u = optwalk_TC(wstar, nsteps, boundaryvels = (wstar.vm, wstar.vm), max_P=max_P, boundarywork=false, J="u");
end
# Perturbed gait
p_u = optwalk_TC(wstar, nsteps, boundaryvels = (wstar.vm, wstar.vm), max_P=max_P, boundarywork=false, J="u", perts = [1 1 pert_mag 1 1 1 1]);
p_dv = optwalk_TC(wstar, nsteps, boundaryvels = (wstar.vm, wstar.vm), max_P=max_P, boundarywork=false, J="dv", perts = [1 1 pert_mag 1 1 1 1]);


In [42]:
using MAT
if group == 1  
    matwrite("NS_proactive_u.mat", Dict("NS_proactive_u" => p_u))
elseif group == 2
    matwrite("S20_proactive_u.mat", Dict("S20_proactive_u" => p_u))
elseif group == 3
    matwrite("S40_proactive_u.mat", Dict("S40_proactive_u" => p_u))
elseif group == 4
    matwrite("S60_proactive_u.mat", Dict("S60_proactive_u" => p_u))
elseif group == 5
    matwrite("S0_proactive_u.mat", Dict("S0_proactive_u" => p_u))
end

LoadError: This is the write function for CompositeKind, but the input doesn't fit

In [43]:
using MAT

if group == 1  
    matwrite("NS_proactive_dv.mat", Dict("NS_proactive_dv" => p_dv))
elseif group == 2
    matwrite("S20_proactive_dv.mat", Dict("S20_proactive_dv" => p_dv))
elseif group == 3
    matwrite("S40_proactive_dv.mat", Dict("S40_proactive_dv" => p_dv))
elseif group == 4
    matwrite("S60_proactive_dv.mat", Dict("S60_proactive_dv" => p_dv))
elseif group == 5
    matwrite("S0_proactive_dv.mat", Dict("S0_proactive_dv" => p_dv))
end

LoadError: This is the write function for CompositeKind, but the input doesn't fit

In [44]:
using MAT

if group == 1  
    matwrite("NS_nominal_u.mat", Dict("NS_nominal_u" => nom_u))
elseif group == 2
    matwrite("S20_nominal_u.mat", Dict("S20_nominal_u" => nom_u))
elseif group == 3
    matwrite("S40_nominal_u.mat", Dict("S40_nominal_u" => nom_u))
elseif group == 4
    matwrite("S60_nominal_u.mat", Dict("S60_nominal_u" => nom_u))
elseif group == 5
    matwrite("S0_nominal_u.mat", Dict("S0_nominal_u" => nom_u))
end

LoadError: This is the write function for CompositeKind, but the input doesn't fit

In [18]:
# Naive control: Apply nominal controls to perturbation condition
naive = multistep(wstar, Ps=nom_u.steps.P, perts=[1 1 pert_mag 1 1 1 1])

# Reactive control: No anticipatory control - just react after the perturbation happens
reactive_1 = multistep(wstar, Ps=nom_u.steps.P[1:3], perts = [1 1 pert_mag])
reactive_2u = optwalk_TC(wstar, 4, boundaryvels=(reactive_1.steps[end].vm,wstar.vm), max_P=max_P[4:7], boundarywork=false, perts=[1 1 1 1], J="u", avg_speed=0)
reactive_u = cat(reactive_1, reactive_2u);

In [46]:
using MAT

if group == 1  
    matwrite("NS_naive.mat", Dict("NS_naive" => naive))
elseif group == 2
    matwrite("S20_naive.mat", Dict("S20_naive" => naive))
elseif group == 3
    matwrite("S40_naive.mat", Dict("S40_naive" => naive))
elseif group == 4
    matwrite("S60_naive.mat", Dict("S60_naive" => naive))
elseif group == 5
    matwrite("S0_naive.mat", Dict("S0_naive" => naive))
end


LoadError: This is the write function for CompositeKind, but the input doesn't fit

In [19]:
using MAT

if group == 1  
    matwrite("NS_reactive_u.mat", Dict("NS_reactive_u" => reactive_u))
elseif group == 2
    matwrite("S20_reactive_u.mat", Dict("S20_reactive_u" => reactive_u))
elseif group == 3
    matwrite("S40_reactive_u.mat", Dict("S40_reactive_u" => reactive_u))
elseif group == 4
    matwrite("S60_reactive_u.mat", Dict("S60_reactive_u" => reactive_u))
elseif group == 5
    matwrite("S0_reactive_u.mat", Dict("S0_reactive_u" => reactive_u))
end

LoadError: This is the write function for CompositeKind, but the input doesn't fit

In [20]:
# No anticipatory control - just react after the perturbation happens
reactive_1 = multistep(wstar, Ps=nom_u.steps.P[1:3], perts = [1 1 pert_mag])
reactive_2dv = optwalk_TC(wstar, 4, boundaryvels=(reactive_1.steps[end].vm,wstar.vm), max_P = max_P[4:7], boundarywork=false, perts=[1 1 1 1], J="dv", avg_speed=0)
reactive_dv = cat(reactive_1, reactive_2dv);

using MAT

if group == 1  
    matwrite("NS_reactive_dv.mat", Dict("NS_reactive_dv" => reactive_dv))
elseif group == 2
    matwrite("S20_reactive_dv.mat", Dict("S20_reactive_dv" => reactive_dv))
elseif group == 3
    matwrite("S40_reactive_dv.mat", Dict("S40_reactive_dv" => reactive_dv))
elseif group == 4
    matwrite("S60_reactive_dv.mat", Dict("S60_reactive_dv" => reactive_dv))
elseif group == 5
    matwrite("S0_reactive_dv.mat", Dict("S0_reactive_dv" => reactive_dv))
end

LoadError: This is the write function for CompositeKind, but the input doesn't fit